In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../results/olmo2-1b-base_sample50.csv")
print(df.head())

                      subject answer       p_A       p_B       p_C       p_D  \
0     high_school_mathematics      C  0.343887  0.208578  0.251593  0.195941   
1                 human_aging      B  0.212073  0.289869  0.272307  0.225751   
2              moral_disputes      C  0.198884  0.239900  0.289374  0.271842   
3  high_school_macroeconomics      D  0.176992  0.257522  0.351991  0.213494   
4  high_school_microeconomics      D  0.132909  0.193381  0.264321  0.409389   

   pred  confidence  is_correct  letter_mass  
0     0    0.343887       False     0.825352  
1     1    0.289869        True     0.880412  
2     2    0.289374        True     0.851003  
3     2    0.351991       False     0.830658  
4     3    0.409389        True     0.825864  


In [6]:
confidences = df["confidence"].to_numpy()
correct = df["is_correct"].to_numpy().astype(int)
probs = df[["p_A", "p_B", "p_C", "p_D"]].to_numpy()
answers = df["answer"].map({"A": 0, "B": 1, "C": 2, "D": 3}).to_numpy()

In [7]:
def compute_ece(confidences, correct, bin_number=10):
    edges = np.linspace(0, 1, bin_number + 1)
    ece = 0
    for bin in range(0, bin_number):
        lo, hi = edges[bin], edges[bin + 1]
        in_bin = (confidences > lo) & (confidences <= hi)
        if in_bin.sum() == 0:
            continue
        bin_acc = correct[in_bin].mean()
        bin_confidence = confidences[in_bin].mean()
        weight = in_bin.mean()
        ece += weight * abs(bin_acc - bin_confidence)
    return ece

In [8]:
def compute_brier(probs, answers):
    truth = np.eye(4)[answers]
    squared = (probs - truth) ** 2
    per_question = squared.sum(axis = 1)
    return per_question.mean()

In [10]:
print(f"Accuracy: {correct.mean():.3f}")
print(f"Mean Confidence: {confidences.mean():.3f}")
print(f"ECE: {compute_ece(confidences, correct):.3f}")
print(f"Brier: {compute_brier(probs, answers):.3f}")


Accuracy: 0.420
Mean Confidence: 0.364
ECE: 0.065
Brier: 0.725
